In [ ]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, widgets, Layout
from IPython.display import display, HTML

def calculate_and_plot_gimbal(
    m_n, L_n, R, m_b, L_b, r, m_m, L_m, R_m, r_m,
    m_B, H_B, R_B1, R_B2, m_w, r_w, d_w,
    m_G, H_G, R_G1, R_G2, m_U, I_U_cm, d_U,
    d_x, theta_deg
):
    x_n = 0.75 * L_n
    x_b = L_n + 0.5 * L_b
    x_m = L_n + L_b - 0.5 * L_m
    M_R = m_n + m_b + m_m
    x_cg = ((m_n * x_n) + (m_b * x_b) + (m_m * x_m)) / M_R
    
    I_cm_n = m_n * ((3/20)*R**2 + (3/80)*L_n**2)
    I_cm_b = (1/12) * m_b * (3*(R**2 + r**2) + L_b**2)
    I_cm_m = (1/12) * m_m * (3*(R_m**2 + r_m**2) + L_m**2)
    
    I_R_lateral = (I_cm_n + m_n*(x_n - x_cg)**2) + (I_cm_b + m_b*(x_b - x_cg)**2) + (I_cm_m + m_m*(x_m - x_cg)**2)
    
    I_R_z = 0.5 * M_R * R**2
    I_B_z = 0.5 * m_B * (R_B1**2 + R_B2**2)
    I_w_z_eff = 2 * m_w * R_B2**2
    I_Total_Z = I_R_z + I_B_z + I_w_z_eff
    
    I_B_y = m_B * ((1/12)*H_B**2 + 0.25*(R_B1**2 + R_B2**2))
    I_G_y = m_G * ((1/12)*H_G**2 + 0.25*(R_G1**2 + R_G2**2))
    I_w_y = 4 * ((0.25 * m_w * r_w**2) + m_w * d_w**2)
    I_Total_Y = I_R_lateral + I_B_y + I_G_y + I_w_y
    
    theta = np.radians(theta_deg)
    I_U_x_shifted = I_U_cm + m_U * d_U**2
    I_internal_local = I_Total_Y * (np.cos(theta)**2) + I_Total_Z * (np.sin(theta)**2)
    M_internal = M_R + m_B + m_G + 4 * m_w
    I_internal_shifted = I_internal_local + M_internal * d_x**2
    I_Total_X = I_U_x_shifted + I_internal_shifted
    
    pct_increase_Y = ((I_Total_Y - I_R_lateral) / I_R_lateral) * 100
    pct_increase_X = ((I_Total_X - I_R_lateral) / I_R_lateral) * 100
    
    status_Y = "PASS" if pct_increase_Y <= 5.0 else "FAIL"
    status_X = "PASS" if pct_increase_X <= 5.0 else "FAIL"
    color_Y = "green" if status_Y == "PASS" else "red"
    color_X = "green" if status_X == "PASS" else "red"
    
    html_output = f"""
    <div style='border: 2px solid #ccc; padding: 15px; border-radius: 8px; background-color: #f9f9f9; font-family: Arial;'>
        <h3 style='margin-top:0;'>Gimbal Inertia Verification Report</h3>
        <p><b>Baseline Rocket Inertia:</b> {I_R_lateral:.6f} kg·m²</p>
        <hr>
        <p><b>Axis 2 (Y-Axis Pins) Total Inertia:</b> {I_Total_Y:.6f} kg·m²</p>
        <p>Inertia Increase: <span style='color:{color_Y}; font-weight:bold;'>{pct_increase_Y:.2f}% ({status_Y})</span> (Target: ≤ 5.0%)</p>
        <hr>
        <p><b>Axis 3 (X-Axis Main Drive) Total Inertia at {theta_deg}°:</b> {I_Total_X:.6f} kg·m²</p>
        <p>Inertia Increase: <span style='color:{color_X}; font-weight:bold;'>{pct_increase_X:.2f}% ({status_X})</span> (Target: ≤ 5.0%)</p>
    </div>
    """
    display(HTML(html_output))
    
    fig = go.Figure()
    z_cyl = np.linspace(-L_b/2, L_b/2, 20)
    theta_cyl = np.linspace(0, 2*np.pi, 20)
    theta_mesh, z_mesh = np.meshgrid(theta_cyl, z_cyl)
    x_mesh = R * np.cos(theta_mesh)
    y_mesh = R * np.sin(theta_mesh)
    
    x_rot = x_mesh
    y_rot = y_mesh * np.cos(theta) - z_mesh * np.sin(theta)
    z_rot = y_mesh * np.sin(theta) + z_mesh * np.cos(theta) + d_x
    
    fig.add_trace(go.Surface(x=x_rot, y=y_rot, z=z_rot, colorscale='Blues', showscale=False, opacity=0.6))
    fig.update_layout(
        title=f"3D Skeleton Position Preview (Tilt: {theta_deg}°)",
        scene=dict(xaxis=dict(range=[-0.5, 0.5]), yaxis=dict(range=[-0.5, 0.5]), zaxis=dict(range=[-0.5, 0.5 + d_x])),
        margin=dict(l=0, r=0, b=0, t=40), width=500, height=400
    )
    fig.show()

style = {'description_width': 'initial'}
layout = Layout(width='350px')

interact(
    calculate_and_plot_gimbal,
    m_n=widgets.FloatSlider(value=0.03, min=0.01, max=0.10, step=0.005, description='Nose Cone Mass (kg)', style=style, layout=layout),
    L_n=widgets.FloatSlider(value=0.15, min=0.05, max=0.30, step=0.01, description='Nose Cone Length (m)', style=style, layout=layout),
    R=widgets.FloatSlider(value=0.025, min=0.01, max=0.08, step=0.005, description='Rocket Radius (m)', style=style, layout=layout),
    m_b=widgets.FloatSlider(value=0.15, min=0.05, max=0.50, step=0.01, description='Body Tube Mass (kg)', style=style, layout=layout),
    L_b=widgets.FloatSlider(value=0.50, min=0.20, max=1.20, step=0.05, description='Body Tube Length (m)', style=style, layout=layout),
    r=widgets.FloatSlider(value=0.023, min=0.005, max=0.075, step=0.005, description='Body Inner Rad (m)', style=style, layout=layout),
    m_m=widgets.FloatSlider(value=0.10, min=0.05, max=0.25, step=0.01, description='Motor Mass (kg)', style=style, layout=layout),
    L_m=widgets.FloatSlider(value=0.07, min=0.04, max=0.15, step=0.01, description='Motor Length (m)', style=style, layout=layout),
    R_m=widgets.FloatSlider(value=0.012, min=0.005, max=0.02),
    r_m=widgets.FloatSlider(value=0.004, min=0.0, max=0.01),
    m_B=widgets.FloatSlider(value=0.40, min=0.05, max=2.0, step=0.05, description='Blue Ring Mass (kg)', style=style, layout=layout),
    H_B=widgets.FloatSlider(value=0.08, min=0.02, max=0.20),
    R_B1=widgets.FloatSlider(value=0.05, min=0.02, max=0.15),
    R_B2=widgets.FloatSlider(value=0.06, min=0.03, max=0.20),
    m_w=widgets.FloatSlider(value=0.02, min=0.005, max=0.10),
    r_w=widgets.FloatSlider(value=0.015, min=0.005, max=0.05),
    d_w=widgets.FloatSlider(value=0.07, min=0.03, max=0.25),
    m_G=widgets.FloatSlider(value=0.60, min=0.10, max=2.5, step=0.05, description='Gold Ring Mass (kg)', style=style, layout=layout),
    H_G=widgets.FloatSlider(value=0.09, min=0.02, max=0.20),
    R_G1=widgets.FloatSlider(value=0.07, min=0.04, max=0.25),
    R_G2=widgets.FloatSlider(value=0.08, min=0.05, max=0.30),
    m_U=widgets.FloatSlider(value=1.20, min=0.20, max=5.0, step=0.1, description='U-Bracket Mass (kg)', style=style, layout=layout),
    I_U_cm=widgets.FloatSlider(value=0.015, min=0.001, max=0.10),
    d_U=widgets.FloatSlider(value=0.15, min=0.05, max=0.40),
    d_x=widgets.FloatSlider(value=0.20, min=0.05, max=0.50, description='Grey Rod Offset dx (m)', style=style, layout=layout),
    theta_deg=widgets.IntSlider(value=0, min=-45, max=45, step=5, description='Test Tilt Angle (deg)', style=style, layout=layout)
);